In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
basepath = Path("./").resolve().parent / "runs"
basepath

In [ ]:
detect_paths = ["detect001", "detect005", "detect01", "detect02"]

model_paths = [
    "yoloe_rodentornot",
    "yoloe_animalornot",
    "yolo26",
    "speciesnet",
    "biotrove-clip_species_names",
    "biotrove-clip_rodentornot",
    "biotrove-clip_common_names",
    "biotrove-clip_animalornot",
]

species_classes = [
    "Rattus norvegicus",
    "Rattus rattus",
    "Mus musculus",
    "Myodes glareolus",
    "Apodemus agrarius",
    "Apodemus flavicollis",
    "Apodemus sylvaticus",
    "Arvicola amphibius",
    "Microtus arvalis",
    "Microtus agrestis",
    "Crocidura leucodon",
]

common_name_classes = [
    "brown rat",
    "black rat",
    "house mouse",
    "bank vole",
    "striped field mouse",
    "yellow-necked mouse",
    "wood mouse",
    "european water vole",
    "common vole",
    "field vole",
    "bicolored white-toothed shrew",
]

animal_classes_yolo26 = [
    "bird",
    "bear",
    "cat",
    "dog",
    "sheep",
    "elephant",
    "cow",
    "giraffe",
    "horse",
    "zebra",
]

animal_or_not = [
    "This is a photo with an animal like a mouse, rat, cat, fox, or hamster in it",
    "This is a photo without any animal at all in it, but a photo of something else entirely like trash, a car, a plastic box or a piece of wood or a leaf",
]

rodent_or_not = [
    "This is a photo with a rodent like a rat, mouse or vole, or similar small mammal like a shrew in it",
    "This is a photo without a rodent in it at all, but a photo of some other animal like a snake, bird or human",
]

# when it has found these it has found the animal in the picture
yolo_animal_found = ["animal", "bear", "cat", "sheep", "cow", "dog", "bird"]

# when it reports these it has found the animal in the picture
speciesnet_non_animal = ["human", "vehicle", "no cv result", "blank"]


In [ ]:
full_data = []
for dp in detect_paths:
    for mp in model_paths:
        path = Path(basepath) / dp / mp / "central-europe"

        for sp in path.iterdir():

            if ".DS_Store" in str(sp):
                continue
            results = json.loads((sp / "detections.json").read_text())

            for imgname, detects in results.items():
                # find the entry with the max conf
                if len(detects) == 0:
                    cls = "no cv result"
                    conf= 1.0
                    aggcls = cls
                    if mp == "yolo26":
                        if cls in animal_classes_yolo26:
                            aggcls = "animal"
                        else:
                            aggcls = "nonanimal"

                    if mp == "speciesnet":
                        if cls not in speciesnet_non_animal:
                            aggcls = "animal"
                        else:
                            aggcls = "nonanimal"

                    full_data.append([
                            dp, mp, sp.name, imgname, cls, aggcls, conf,
                    ])
                else:

                    for entry in detects:
                        cls = entry["class"]
                        aggcls = cls
                        conf = entry["conf"]
                        if mp == "yolo26":
                            if cls in animal_classes_yolo26:
                                aggcls = "animal"
                            else:
                                aggcls = "nonanimal"

                        if mp == "speciesnet":
                            if cls not in speciesnet_non_animal:
                                aggcls = "animal"
                            else:
                                aggcls = "nonanimal"

                        full_data.append([
                                dp, mp, sp.name, imgname, cls, aggcls, conf,
                        ])

In [ ]:
full_data = pd.DataFrame(
    full_data,
    columns = [
        "detect_path", "model", "species", "image", "class", "aggregated_class", "confidence"
    ]
)
full_data

In [ ]:
len(list(Path("/home/hmack/Development/rodent_experiments/datasets/biotrove-central-europe/filtered/Apodemus agrarius").iterdir()))

In [ ]:
full_data["class_canonical"] = (
    full_data["class"]
        .str.replace(r"^This is a photo\s+", "", regex=True)
        .str.replace(r"^This is not a photo\s+", "", regex=True)
        .str.strip()
)

In [ ]:
full_data.head()

In [ ]:
full_data.to_csv(basepath / "full_data.csv")

### get best detections 

In [ ]:
maxconf_data = full_data.loc[
    full_data.groupby(["detect_path", "model", "species", "image"])["confidence"].idxmax()
].reset_index(drop=True)
maxconf_data

In [ ]:
maxconf_data.to_csv(basepath / "maxconf_data.csv")

# Visualizations

We always use the most confident detections for evaluations

## Rodent or not - YoloE and Biotrove-clip

### confusion matrix

all images contain a rodent, i.e., there are 0 true non-detections and 100% true detections

In [ ]:
subset = ["biotrove-clip_rodentornot", "yoloe_rodentornot"]
sub_df = maxconf_data[maxconf_data["model"].isin(subset)]
detect_paths = sub_df.loc[:, "detect_path"].unique()

for dp in detect_paths:
    print(dp)
    ax_idx = 0
    fig, axs = plt.subplots(5, 4, figsize=(24, 20))
    axs = axs.flatten()
    dpdf = sub_df[
        sub_df["detect_path"].isin(
            [
                dp,
            ]
        )
    ]

    for key, grp in dpdf.groupby(["model", "species"]):
        y_pred = grp.loc[:, "class_canonical"].to_list()

        true_key = (
            rodent_or_not[0].replace("This is a photo", "", ).replace("This is not a photo", "").strip()
        )

        false_key = rodent_or_not[1].replace("This is a photo", "", ).replace("This is not a photo", "").strip()

        y_true = [true_key for _ in range(len(y_pred))]

        labels=[true_key, false_key]
        display_labels = [
            "rodent",
            "not rodent"
        ]

        title = f"{key[0]}, {key[1]}"
        confmat = confusion_matrix(
            y_true,
            y_pred,
            labels=labels,          # real labels used in your data
            normalize="all",
        )
        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
            display_labels=display_labels,  # short labels shown on plot
        )

        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(title)
        ax_idx += 1
    fig.suptitle(dp)
    fig.tight_layout(rect=[0, 0, 1, 0.95])


it looks like none of these systems is able to reliable detect rodents with this prompt and dataset and calling procedure. 

- prompt engineering? 
- bug in call/computation procedure? 

## Animal or not YoloE and biotrove

### confusion matrix

There is always an animal in the image, so 100% true detections and 0% non-detections

In [ ]:
subset = ["yoloe_animalornot", "biotrove-clip_animalornot"]
sub_df = full_data[full_data["model"].isin(subset)]
detect_paths = sub_df.loc[:, "detect_path"].unique()
print(detect_paths)
for dp in detect_paths:
    print(dp)
    ax_idx = 0
    fig, axs = plt.subplots(5, 4, figsize=(24, 20))
    axs = axs.flatten()
    dpdf = sub_df[
        sub_df["detect_path"].isin(
            [
                dp,
            ]
        )
    ]

    # FIXME: this is so stupid and complicated!!
    for key, grp in dpdf.groupby(["model", "species"]):
        y_pred = grp.loc[:, "class_canonical"].to_list()

        true_key = (
            animal_or_not[0].replace("This is a photo", "", ).replace("This is not a photo", "").strip()
        )
        false_key = animal_or_not[1].replace("This is a photo", "", ).replace("This is not a photo", "").strip()

        y_true = [true_key for _ in range(len(y_pred))]

        labels=[true_key, false_key]
        display_labels = [
            "animal",
            "not animal"
        ]

        title = f"{key[0]}, {key[1]}"
        confmat = confusion_matrix(
            y_true,
            y_pred,
            labels=labels,          # real labels used in your data
            normalize="all",
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
            display_labels=display_labels,  # short labels shown on plot
        )

        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(title)
        ax_idx += 1
    fig.suptitle(dp)
    fig.tight_layout(rect=[0, 0, 1, 0.95])


## Accuracy of species name identification with Biotrove-clip

In [ ]:
subset = ["biotrove-clip_species_names"]
sub_df = full_data[full_data["model"].isin(subset)].copy()
detect_paths = sub_df.loc[:, "detect_path"].unique()
for dp in detect_paths:
    ax_idx = 0
    fig, axs = plt.subplots(3, 4, figsize=(24, 18))
    axs = axs.flatten()
    dpdf = sub_df[sub_df["detect_path"].eq(dp)]

    species_labels = []
    for key, _ in dpdf.groupby(["model", "species"]):
        _, species = key
        species_labels.append(species)

    for key, grp in dpdf.groupby(["model", "species"]):
        model, species = key
        y_pred = grp.loc[:, "class"].to_list()
        y_true = [species for _ in range(len(y_pred))]

        confmat = confusion_matrix(
            y_true,
            y_pred,
            normalize=None,
            labels=species_labels
        )
        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
            display_labels=species_labels
        )
        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(f"{model}, {species}")
        axs[ax_idx].tick_params(axis="x", labelrotation=90)
        ax_idx += 1

    for ax in axs[ax_idx:]:
        ax.set_visible(False)

    fig.suptitle(f"Biotrove-CLIP species-name predictions: {dp}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])


## Accuracy of common name identification with Biotrove-clip

In [ ]:
subset = ["biotrove-clip_common_names",]
sub_df = full_data[full_data["model"].isin(subset)].copy()
common_name_lookup = dict(zip(species_classes, common_name_classes))

detect_paths = sub_df.loc[:, "detect_path"].unique()
for dp in detect_paths:
    ax_idx = 0
    fig, axs = plt.subplots(3, 4, figsize=(24, 18))
    axs = axs.flatten()
    dpdf = sub_df[sub_df["detect_path"].eq(dp)]

    species_labels = []
    for key, _ in dpdf.groupby(["model", "species"]):
        _, species = key
        species_labels.append(common_name_lookup[species])

    for key, grp in dpdf.groupby(["model", "species"]):
        model, species = key
        y_pred = grp.loc[:, "class_canonical"].to_list()
        y_true = [common_name_lookup[species] for _ in range(len(y_pred))]

        confmat = confusion_matrix(
                y_true,
                y_pred,
                normalize=None,
                labels = species_labels
            )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
            display_labels=species_labels
        )
        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(f"{model}, {species}")
        axs[ax_idx].tick_params(axis="x", labelrotation=90)
        ax_idx += 1

    for ax in axs[ax_idx:]:
        ax.set_visible(False)

    fig.suptitle(f"Biotrove-CLIP common-name predictions: {dp}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])


## Speciesnet animal identification accuracy
We might be able to identify animals with speciesnet, then train a yolo model on its bounding boxes (**check license first if this is permissible**), and then use that to get good animal crops but with a more accessible API. Yolo is far stronger in that regard than speciesnet. Alternatively, if speciesnet would work, we could fine tune it too. **Check if speciesnet can be retrained or finetuned**

### confusion matrix animal aggregation

In [ ]:
subset = ['speciesnet',]
sub_df = full_data[full_data["model"].isin(subset)].copy()

detect_paths = sub_df.loc[:, "detect_path"].unique()
for dp in detect_paths:
    ax_idx = 0
    fig, axs = plt.subplots(3, 4, figsize=(24, 18))
    axs = axs.flatten()
    dpdf = sub_df[sub_df["detect_path"].eq(dp)]

    for key, grp in dpdf.groupby(["model", "species"]):
        model, species = key
        y_pred = grp.loc[:, "aggregated_class"].to_list()
        y_true = ["animal" for _ in range(len(y_pred))]

        labels = ["animal", "no animal"]
        confmat = confusion_matrix(
            y_true,
            y_pred,
            labels=labels,
            normalize=None,
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
            display_labels=labels,
        )
        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(f"{model}, {species}")
        ax_idx += 1

    for ax in axs[ax_idx:]:
        ax.set_visible(False)

    fig.suptitle(f"SpeciesNet animal/no-animal predictions: {dp}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])


I am not sure this is a good idea, b/c this is  purely based on the observed performance of the model, and it's not known that animal and non-animal classes clearly separate in laten space in a way that would make this consolidation meaningful in general

I was unsuccessful atm to change the confidence limit for speciesnet. however, it is rather confident in animal predictions, so extracting speciesnet crops could work well. Need to look it up better, docs are not great compared to YOLO

In [ ]:
subset = ["speciesnet",]
sub_df = full_data[full_data["model"].isin(subset)]

detect_paths = sub_df.loc[:, "detect_path"].unique()
for dp in detect_paths:
    ax_idx = 0
    fig, axs = plt.subplots(3, 4, figsize=(24, 18))
    axs = axs.flatten()
    dpdf = sub_df[sub_df["detect_path"].eq(dp)]


    for key, grp in dpdf.groupby(["model", "species"]):
        model, species = key
        y_pred = grp.loc[:, "class_canonical"].to_list()
        y_true = grp.loc[:, "species"].to_list()

        confmat = confusion_matrix(
            y_true,
            y_pred,
            normalize=None,
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
        )
        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(f"{model}, {species}")
        ax_idx += 1

    for ax in axs[ax_idx:]:
        ax.set_visible(False)

    fig.suptitle(f"SpeciesNet species predictions: {dp}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])


## YOLO animal identification 

I tried to put together all possible animal like classes, but that's not a reliable system b/c it assumes that detections of a rodent is close to the detection of other animals in the latent space of the model, which for nonlinear systems is not a given a priori, and the whole idea is based on the empirical observations that it classifiers rodents it sees as animal like things (bear, cat...)

## confusion matrix

In [ ]:
subset = ["yolo26",]
sub_df = full_data[full_data["model"].isin(subset)].copy()

detect_paths = sub_df.loc[:, "detect_path"].unique()
for dp in detect_paths:
    ax_idx = 0
    fig, axs = plt.subplots(3, 4, figsize=(24, 18))
    axs = axs.flatten()
    dpdf = sub_df[sub_df["detect_path"].eq(dp)]

    for key, grp in dpdf.groupby(["model", "species"]):
        model, species = key
        y_pred = grp.loc[:, "aggregated_class"].to_list()
        y_true = ["animal" for _ in range(len(y_pred))]

        labels = ["animal", "nonanimal"]
        display_labels = ["animal", "not animal"]
        confmat = confusion_matrix(
            y_true,
            y_pred,
            labels=labels,
            normalize="all",
        )

        disp = ConfusionMatrixDisplay(
            confusion_matrix=confmat,
            display_labels=display_labels,
        )
        disp.plot(
            ax=axs[ax_idx],
            cmap="Blues",
            colorbar=False,
            values_format=".2f",
        )
        axs[ax_idx].set_title(f"{model}, {species}")
        ax_idx += 1

    for ax in axs[ax_idx:]:
        ax.set_visible(False)

    fig.suptitle(f"YOLO26 animal/non-animal predictions: {dp}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
